# Lab 3: S3 API & SDK (boto3)

## 🎯 Mục tiêu
- Cấu hình **boto3** kết nối tới MinIO (endpoint, credentials)
- Thực hiện **PutObject**, **GetObject**, **ListObjectsV2**
- Upload/download file và dữ liệu từ Python

## 📋 Prerequisites
- MinIO đang chạy: `docker compose up -d`
- Đã cài boto3: `pip install boto3` hoặc `./setup_s3_lab.sh`


## 1. Tạo S3 client trỏ tới MinIO

In [ ]:
import boto3
from botocore.client import Config
import io

ENDPOINT = "http://localhost:9000"
ACCESS_KEY = "minioadmin"
SECRET_KEY = "minioadmin"
REGION = "us-east-1"
BUCKET = "lab-bucket"

s3 = boto3.client(
    "s3",
    endpoint_url=ENDPOINT,
    aws_access_key_id=ACCESS_KEY,
    aws_secret_access_key=SECRET_KEY,
    region_name=REGION,
    config=Config(signature_version="s3v4"),
)

print("S3 client đã cấu hình (endpoint:", ENDPOINT, ")")

## 2. Tạo bucket (nếu chưa có)

In [ ]:
try:
    s3.create_bucket(Bucket=BUCKET)
    print(f"✅ Bucket '{BUCKET}' đã tạo.")
except Exception as e:
    if "BucketAlreadyOwnedByYou" in str(type(e).__name__) or "BucketAlreadyExists" in str(e):
        print(f"Bucket '{BUCKET}' đã tồn tại.")
    else:
        raise e

## 3. PutObject – Upload dữ liệu

In [ ]:
# Upload string như object
key_hello = "demo/hello.txt"
s3.put_object(
    Bucket=BUCKET,
    Key=key_hello,
    Body=b"Hello from S3 Lab - MinIO!",
    ContentType="text/plain",
)
print(f"✅ Đã upload: {key_hello}")

In [ ]:
# Upload dữ liệu dạng CSV (từ pandas hoặc string)
import csv

key_csv = "data/sample.csv"
csv_content = "name,value\nalice,100\nbob,200\ncharlie,300"
s3.put_object(
    Bucket=BUCKET,
    Key=key_csv,
    Body=csv_content.encode("utf-8"),
    ContentType="text/csv",
)
print(f"✅ Đã upload: {key_csv}")

## 4. ListObjectsV2 – Liệt kê objects

In [ ]:
paginator = s3.get_paginator("list_objects_v2")
for page in paginator.paginate(Bucket=BUCKET):
    for obj in page.get("Contents", []):
        print(f"  {obj['Key']}  ({obj['Size']} bytes)")

# List với prefix (giống "thư mục")
print("\nObjects dưới prefix 'data/':")
for page in paginator.paginate(Bucket=BUCKET, Prefix="data/"):
    for obj in page.get("Contents", []):
        print(f"  {obj['Key']}")

## 5. GetObject – Download dữ liệu

In [ ]:
response = s3.get_object(Bucket=BUCKET, Key=key_hello)
body = response["Body"].read()
print("Nội dung hello.txt:", body.decode("utf-8"))

In [ ]:
# Đọc CSV từ S3 vào pandas
import pandas as pd

response = s3.get_object(Bucket=BUCKET, Key=key_csv)
df = pd.read_csv(io.BytesIO(response["Body"].read()))
print(df)

## 6. DeleteObject (tùy chọn)

In [ ]:
# Bỏ comment để xóa object
# s3.delete_object(Bucket=BUCKET, Key=key_hello)
# print("Đã xóa", key_hello)

## 7. Tóm tắt

- **endpoint_url** trỏ tới MinIO (http://localhost:9000)
- **PutObject**: Body có thể là bytes hoặc file-like object
- **ListObjectsV2** + **Prefix**: list theo "thư mục"
- **GetObject**: đọc body qua `response['Body'].read()`

**Tiếp theo**: Lab 4 – Dùng MinIO làm storage backend cho data pipeline (Iceberg/s3fs).